### ЗАДАЧА: Панель KYC-верификации продавцов

Compliance-команда маркетплейса проверяет продавцов перед допуском к выплатам.
Для каждого кейса верификации нужно хранить:
- `case_id`
- `seller`
- `country`
- `risk_level`
- `status`
- `reviewer`
- `docs_uploaded`
- `decision`

Нужно реализовать внутреннюю консольную панель по паттерну `MVC`, где:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за отображение;
- `Controller` принимает действия оператора и связывает слои.

Сценарий работы:
- новые кейсы создаются в статусе `new`;
- если документов не хватает, кейс переводится в `docs_pending`;
- когда документы получены, кейс переводится в `docs_received` и `docs_uploaded = True`;
- после этого кейс можно взять в `reviewing`;
- из `reviewing` кейс можно завершить как `approved`, `rejected` или `escalated`.

Бизнес-правила:
- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить reviewer несуществующему кейсу;
- нельзя запросить документы у финального кейса (`approved`, `rejected`, `escalated`);
- нельзя отметить документы полученными, если кейс не в статусе `docs_pending`;
- нельзя начать review без назначенного reviewer;
- нельзя начать review, если `docs_uploaded == False`;
- нельзя перевести кейс в `reviewing`, если он не находится в статусе `docs_received`;
- нельзя завершить кейс как `approved`, `rejected` или `escalated`, если он не в статусе `reviewing`;
- финальный кейс нельзя изменять дальше.

НЕОБХОДИМО:

1. Реализовать сущность `KycCase`.
2. Реализовать `KycModel`, который умеет:
   - добавлять кейс,
   - назначать reviewer,
   - запрашивать документы,
   - отмечать документы как полученные,
   - переводить кейс в `reviewing`,
   - завершать кейс как `approved`,
   - завершать кейс как `rejected`,
   - завершать кейс как `escalated`,
   - возвращать список кейсов для отображения,
   - возвращать сводку количества кейсов по статусам.
3. Реализовать `KycView`, который умеет:
   - показывать список кейсов,
   - показывать сводку по статусам,
   - показывать успешные сообщения,
   - показывать ошибки.
4. Реализовать `KycController`, который вызывает методы модели и передает результат во view.
5. Загрузить начальные кейсы из `initial_cases` в модель.
6. Обработать все действия из `actions` через `controller`.
7. В конце вывести:
   - финальный список кейсов,
   - сводку по статусам.


In [7]:
from dataclasses import dataclass


initial_cases = [
    ("KYC-100", "best-gadgets", "DE", "high"),
    ("KYC-101", "home-decor-plus", "PL", "medium"),
]

actions = [
    ("show",),
    ("assign", "KYC-100", "Olga"),
    ("request_docs", "KYC-100"),
    ("review", "KYC-100"),
    ("docs_received", "KYC-100"),
    ("review", "KYC-100"),
    ("approve", "KYC-100"),
    ("create", "KYC-102", "sport-zone", "TR", "high"),
    ("assign", "KYC-102", "Max"),
    ("docs_received", "KYC-102"),
    ("review", "KYC-102"),
    ("escalate", "KYC-102"),
    ("show",),
]


@dataclass
class KycCase:
    case_id: str
    seller: str
    country: str
    risk_level: str
    status: str = "new"
    reviewer: str | None = None
    docs_uploaded: bool = False
    decision: str | None = None


class KycModel:
    def __init__(self):
        self.cases = {}
        self.final_statuses = {'appoved','rejected','escalated'}
    
    def get_case(self,case_id:str) -> KycCase:
        if case_id not in self.cases:
            raise ValueError
        return self.cases[case_id]
    
    def add_case(self, case_id: str, seller: str, country: str, risk_level: str) -> None:
        if case_id in self.cases:
            raise ValueError('case already in cases')
        self.cases[case_id] = KycCase(case_id, seller, country, risk_level)

    def assign_reviewer(self, case_id: str, reviewer: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        self.cases[case_id].reviewer = reviewer

    def request_docs(self, case_id: str) -> None:
        # if case_id not in self.cases:
        #     raise ValueError("Case not found")
        # if self.cases[case_id].status == 'appoved':
        #     raise ValueError('Already done')
        # if self.cases[case_id].status == 'rejected':
        #     raise ValueError('Already done')
        # if self.cases[case_id].status == 'escalated':
        #     raise ValueError('Already done')
        
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('Already done')

        self.cases[case_id].status = "docs_pending"
        

    def mark_docs_received(self, case_id: str) -> None:
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('Already done')


        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status != 'docs_pending':
            raise ValueError('case is not waiting documents')

        self.cases[case_id].docs_uploaded = True
        self.cases[case_id].status = "docs_received"



    def start_review(self, case_id: str) -> None:
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('Already done')

        if not self.cases[case_id].reviewer:
            raise ValueError('No reviewer')
        if self.cases[case_id].docs_uploaded == False:
            raise ValueError('docs not uploaded')
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.reviewer is None:
            raise ValueError("Reviewer is required")
        if case.docs_uploaded == False:
            raise ValueError('case not received')
        case.status = "reviewing"

    def approve_case(self, case_id: str) -> None:
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('Already done')

        if case_id not in self.cases:
            raise ValueError("Case not found")
        
        if self.cases[case_id].status != 'reviewing':
            raise ValueError('must be reviewing')
        
        self.cases[case_id].status = "approved"
        self.cases[case_id].decision = "approved"

    def reject_case(self, case_id: str) -> None:
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('Already done')

        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status != 'reviewing':
            raise ValueError('must be reviewing')
        case.status = "rejected"
        case.decision = "rejected"

    def escalate_case(self, case_id: str) -> None:
        if self.cases[case_id].status in self.final_statuses:
            raise ValueError('Already done')
        
        if case_id not in self.cases:
            raise ValueError("Case not found")
        case = self.cases[case_id]
        if case.status != 'reviewing':
            raise ValueError('must be reviewing')
        case.status = "escalated"
        case.decision = "escalated"

    def list_cases(self) -> list[str]:
        rows = []
        for case in self.cases.values():
            rows.append(
                f"{case.case_id} | {case.seller} | {case.country} | {case.risk_level} | {case.status} | {case.reviewer} | {case.docs_uploaded} | {case.decision}"
            )
        return rows

    def status_summary(self) -> dict[str, int]:
        summary = {}
        for case in self.cases.values():
            summary[case.status] = summary.get(case.status, 0) + 1
        return summary


class KycView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        print("KYC queue:")
        for row in rows:
            print(row)

    @staticmethod
    def render_summary(summary: dict[str, int]) -> None:
        print("Status summary:", summary)

    @staticmethod
    def render_success(message: str) -> None:
        print("SUCCESS:", message)

    @staticmethod
    def render_error(message: str) -> None:
        print("ERROR:", message)


class KycController:
    def __init__(self, model: KycModel, view: KycView):
        self.model = model
        self.view = view

    def create_case(self, case_id: str, seller: str, country: str, risk_level: str) -> None:
        try:
            self.model.add_case(case_id, seller, country, risk_level)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as error:
            view.render_error(str(error))

    def assign_reviewer(self, case_id: str, reviewer: str) -> None:
        try:
            self.model.assign_reviewer(case_id, reviewer)
            self.view.render_success(f"Reviewer assigned to {case_id}")
        except ValueError as error:
            view.render_error(str(error))

    def request_docs(self, case_id: str) -> None:
        try:
            self.model.request_docs(case_id)
            self.view.render_success(f"Documents requested for {case_id}")
        except ValueError as error:
            view.render_error(str(error))

    def mark_docs_received(self, case_id: str) -> None:
        try:
            self.model.mark_docs_received(case_id)
            self.view.render_success(f"Documents received for {case_id}")
        except ValueError as error:
            view.render_error(str(error))

    def start_review(self, case_id: str) -> None:
        try:
            self.model.start_review(case_id)
            self.view.render_success(f"Case {case_id} moved to review")
        except ValueError as error:
            self.view.render_error(str(error))

    def approve_case(self, case_id: str) -> None:
        try:
            self.model.approve_case(case_id)
            self.view.render_success(f"Case {case_id} approved")
        except ValueError as error:
            self.view.render_error(str(error))

    def reject_case(self, case_id: str) -> None:
        try:
            self.model.reject_case(case_id)
            self.view.render_success(f"Case {case_id} rejected")
        except ValueError as error:
            self.view.render_error(str(error))

    def escalate_case(self, case_id: str) -> None:
        try:
            self.model.escalate_case(case_id)
            self.view.render_success(f"Case {case_id} escalated")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        self.view.render_cases(self.model.list_cases())
        self.view.render_summary(self.model.status_summary())


model = KycModel()
view = KycView()
controller = KycController(model, view)

for case_id, seller, country, risk_level in initial_cases:
    model.add_case(case_id, seller, country, risk_level)

for action in actions:
    if action[0] == "show":
        controller.show_cases()
    elif action[0] == "create":
        _, case_id, seller, country, risk_level = action
        controller.create_case(case_id, seller, country, risk_level)
    elif action[0] == "assign":
        _, case_id, reviewer = action
        controller.assign_reviewer(case_id, reviewer)
    elif action[0] == "request_docs":
        _, case_id = action
        controller.request_docs(case_id)
    elif action[0] == "docs_received":
        _, case_id = action
        controller.mark_docs_received(case_id)
    elif action[0] == "review":
        _, case_id = action
        controller.start_review(case_id)
    elif action[0] == "approve":
        _, case_id = action
        controller.approve_case(case_id)
    elif action[0] == "reject":
        _, case_id = action
        controller.reject_case(case_id)
    elif action[0] == "escalate" :
        _, case_id = action
        controller.escalate_case(case_id)

print()
print("Финальное состояние:")
controller.show_cases()


KYC queue:
KYC-100 | best-gadgets | DE | high | new | None | False | None
KYC-101 | home-decor-plus | PL | medium | new | None | False | None
Status summary: {'new': 2}
SUCCESS: Reviewer assigned to KYC-100
SUCCESS: Documents requested for KYC-100
ERROR: docs not uploaded
SUCCESS: Documents received for KYC-100
SUCCESS: Case KYC-100 moved to review
SUCCESS: Case KYC-100 approved
SUCCESS: Case KYC-102 created
SUCCESS: Reviewer assigned to KYC-102
ERROR: case is not waiting documents
ERROR: docs not uploaded
ERROR: must be reviewing
KYC queue:
KYC-100 | best-gadgets | DE | high | approved | Olga | True | approved
KYC-101 | home-decor-plus | PL | medium | new | None | False | None
KYC-102 | sport-zone | TR | high | new | Max | False | None
Status summary: {'approved': 1, 'new': 2}

Финальное состояние:
KYC queue:
KYC-100 | best-gadgets | DE | high | approved | Olga | True | approved
KYC-101 | home-decor-plus | PL | medium | new | None | False | None
KYC-102 | sport-zone | TR | high | new 

In [ ]:
# KYC queue:
# KYC-100 | best-gadgets | DE | high | new | None | False | None
# KYC-101 | home-decor-plus | PL | medium | new | None | False | None
# Status summary: {'new': 2}
# SUCCESS: Reviewer assigned to KYC-100
# SUCCESS: Documents requested for KYC-100
# ERROR: Documents are required
# SUCCESS: Documents received for KYC-100
# SUCCESS: Case KYC-100 moved to review
# SUCCESS: Case KYC-100 approved
# SUCCESS: Case KYC-102 created
# SUCCESS: Reviewer assigned to KYC-102
# ERROR: Case is not waiting for documents
# ERROR: Documents are required
# ERROR: Case is not in review
# KYC queue:
# KYC-100 | best-gadgets | DE | high | approved | Olga | True | approved
# KYC-101 | home-decor-plus | PL | medium | new | None | False | None
# KYC-102 | sport-zone | TR | high | new | Max | False | None
# Status summary: {'approved': 1, 'new': 2}
#
# Финальное состояние:
# KYC queue:
# KYC-100 | best-gadgets | DE | high | approved | Olga | True | approved
# KYC-101 | home-decor-plus | PL | medium | new | None | False | None
# KYC-102 | sport-zone | TR | high | new | Max | False | None
# Status summary: {'approved': 1, 'new': 2}